In [ ]:
import os, json
for folder in ["corpus", "partial_reading", "prompt", "extraction", "docs"]:
    os.makedirs(folder, exist_ok=True)

!pip install -q plotly scipy pandas
print("Setup done")

Setup done


# STEP 1: Partial Reading

In [ ]:
import re, random

STORY_PATH = "corpus/gift_of_the_magi.txt"

def load_story():
    """Use corpus/gift_of_the_magi.txt if it exists; otherwise ask for an upload (Colab)."""
    if not os.path.exists(STORY_PATH):
        from google.colab import files
        print("Upload gift_of_the_magi.txt now.")
        uploaded = files.upload()
        name = next(iter(uploaded))
        with open(STORY_PATH, "wb") as f:
            f.write(uploaded[name])
    return open(STORY_PATH, encoding="utf-8").read().strip()

raw_story = load_story()

random.seed(42)                       # keeps the random middle block the same every run
sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n+", raw_story) if s.strip()]
third = len(sents) // 3
parts = [sents[:third], sents[third:2 * third], sents[2 * third:]]
n = max(3, round(0.2 * third))        # block length = 20 % of one part
starts = [0, random.randint(0, len(parts[1]) - n), len(parts[2]) - n]
labels = ["first", "random", "last"]

blocks = [" ".join(parts[i][starts[i]:starts[i] + n]) for i in range(3)]
print(f"{len(sents)} sentences; each part has about {third}; each block has {n} sentences.\n")
for i, b in enumerate(blocks):
    print(f"--- PART {i + 1} ({labels[i]} block, sentences {starts[i] + 1}-{starts[i] + n} of the part) ---")
    print(b, "\n")

157 sentences; each part has about 52; each block has 10 sentences.

--- PART 1 (first block, sentences 1-10 of the part) ---
One dollar and eighty-seven cents. That was all. And sixty cents of it was in pennies. Pennies saved one and two at a time by bulldozing the grocer and the vegetable man and the butcher until one’s cheeks burned with the silent imputation of parsimony that such close dealing implied. Three times Della counted it. One dollar and eighty-seven cents. And the next day would be Christmas. There was clearly nothing to do but flop down on the shabby little couch and howl. So Della did it. Which instigates the moral reflection that life is made up of sobs, sniffles, and smiles, with sniffles predominating. 

--- PART 2 (random block, sentences 41-50 of the part) ---
Jim stopped inside the door, as immovable as a setter at the scent of quail. His eyes were fixed upon Della, and there was an expression in them that she could not read, and it terrified her. It was not ange

In [ ]:
MARKED = {
    1: ["pennies", "cheeks burned with the silent imputation of parsimony",
        "shabby little couch", "sobs, sniffles, and smiles"],
    2: ["it terrified her", "I had my hair cut off and sold",
        "My hair grows awfully fast"],
    3: ["I sold the watch to get the money to buy your combs", "wise men",
        "two foolish children", "the wisest", "They are the magi"],
}

norm_q = lambda s: s.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"').lower()
for i, phrases in MARKED.items():
    for p in phrases:
        assert norm_q(p) in norm_q(blocks[i - 1]), f"Marked phrase not in block {i}: {p}"

methodology = (
    "SAMPLING METHODOLOGY\n"
    "I split the story into three equal parts by sentence count. From each part I read one block of about "
    "one-fifth (20%) of the part: the first block of Part 1, a randomly placed block of Part 2 (seed 42) and "
    "the last block of Part 3.\n\n"
)
out = methodology + "MARKED PHRASES\n"
for i in range(3):
    out += f"\nPART {i + 1} ({labels[i]})\n{blocks[i]}\n\nMarked: " + "; ".join(MARKED[i + 1]) + "\n"
open("partial_reading/partial_reading.txt", "w", encoding="utf-8").write(out)
print("Saved partial_reading/partial_reading.txt")

Saved partial_reading/partial_reading.txt


# STEP 2: Initial Impression

In [ ]:
from IPython.display import Markdown, display

DIMENSIONS = {
    "transactional_shaming": {
        "label": "Transactional shaming",
        "definition": "Money handled as a source of exposure and shame, not only as a means of buying.",
        "subs": {
            "amounts_money": "Sums of money, coins and wages (for example dollars, cents, pennies).",
            "counting_saving_bargaining": "Acts of counting, saving, calculating, bargaining and planning to buy.",
            "shame_and_want": "Words and phrases of shame, shabbiness, poverty and lack.",
        },
    },
    "corporeal_commodification": {
        "label": "Corporeal commodification",
        "definition": "A part of the body treated as a thing that can be priced, sold and exchanged.",
        "subs": {
            "body_as_sellable": "Hair and the words used for it when it is treated as goods.",
            "price_sale_exchange": "Words of price, sale and exchange attached to the transaction.",
            "reaction_to_sale": "The body's and the self's reactions to the sale (fear, panic, self-image).",
        },
    },
    "objects_and_feeling": {
        "label": "Objects and feeling",
        "definition": "Gifts and possessions as carriers of feeling and character.",
        "subs": {
            "gifts_possessions": "The gifts and possessions themselves (watch, chain, combs, presents).",
            "qualities_of_objects": "Qualities given to objects (beautiful, rare, precious, simple and chaste).",
            "feelings_attached": "Feelings that the characters attach to people and objects.",
        },
    },
    "paradoxical_value_inversion": {
        "label": "Paradoxical value inversion",
        "definition": "The narrator turns loss into gain and foolishness into wisdom.",
        "subs": {
            "wisdom_foolishness": "Words of wisdom, foolishness, reason and calculation.",
            "narrator_comments": "The narrator's direct addresses, asides and comments on the telling.",
            "ironic_register": "Mock-heroic, formal or exaggerated phrases that create irony.",
        },
    },
}

RESEARCH_QUESTION = ("How does the narrator's language turn economic loss into emotional value, "
                     "and what does the reader learn to call \"wise\"?")
CLAIM = ("Words of shame, counting and selling are concentrated in the first two stages of the story, "
         "while words of wisdom and the narrator's comments are concentrated in the last stage.")

dim_lines = ""
for i, (k, d) in enumerate(DIMENSIONS.items(), 1):
    dim_lines += f"{i}. **{d['label']}.** {d['definition']}\n"
    for j, (s, text_) in enumerate(d["subs"].items()):
        dim_lines += f"   ({'abc'[j]}) *{s}*: {text_}\n"

IMPRESSION = f"""# Initial impression: "The Gift of the Magi" (O. Henry, 1905)

## Sampling methodology
The story is split into three equal parts. From each part, one block equal to about 20% of that section is read: the first block of Part 1, a random block from Part 2 using a fixed seed (42), and the last block of Part 3. Together, these three blocks cover the opening, the moment when Jim sees Della, and the ending.

## My impression
A reading of the three blocks reveals a tension between love and money, guided by a narrator who constantly addresses the audience. In the first block, money is counted repeatedly—emphasised by phrases like "One dollar and eighty-seven cents" and "pennies"—and is directly linked to physical reactions, causing cheeks to burn and prompting Della to howl. By the second block, the sacrifice has already occurred. Della does not describe the event as a loss, choosing instead to declare that her hair grows quickly, while Jim’s unreadable expression frightens her. In the final block, Jim reveals his own sale, prompting the narrator to shift from a joking tone to a direct address. The narrator labels the couple first as "foolish" and ultimately as "the wisest." This shift marks a distinct turning point, where the very acts that initially resembled poverty and waste are renamed as true wealth.

## One overall concept
The core concept uniting the marked phrases is value inversion, where the narrative language transforms a material loss into emotional value. Through this lens, sacrifice reads as richness, and apparent foolishness is reinterpreted as deep wisdom.

## My positionality
The perspective shaping this analysis is that of a college student from a busy, multilingual Indian city. While academic coursework is conducted entirely in English, it functions as an acquired register rather than the primary language spoken at home. This background naturally draws immediate attention to plot progression and emotional resonance, while rare or highly formal vocabulary is processed secondarily. Consequently, the interpretation remains highly alert to themes of financial constraint and familial obligation within a story about gift-giving, while the finer, ironic nuances of a formal English narrator may initially be less apparent.

## My research problem
Reading from the perspective of a student from an urban, multilingual Indian background, the analysis reveals that the first block treats money as a cause of shame, the second treats the body as a commodity that can be sold, and the third allows the narrator to redefine the nature of wisdom. This framework leads to the central inquiry: {RESEARCH_QUESTION} Consequently, the core thesis to be tested is: {CLAIM}

## My dimensions
{dim_lines}"""

open("initial_impression.md", "w", encoding="utf-8").write(IMPRESSION)
display(Markdown(IMPRESSION))

# Initial impression: "The Gift of the Magi" (O. Henry, 1905)

## Sampling methodology
The story is split into three equal parts. From each part, one block equal to about 20% of that section is read: the first block of Part 1, a random block from Part 2 using a fixed seed (42), and the last block of Part 3. Together, these three blocks cover the opening, the moment when Jim sees Della, and the ending.

## My impression
A reading of the three blocks reveals a tension between love and money, guided by a narrator who constantly addresses the audience. In the first block, money is counted repeatedly—emphasised by phrases like "One dollar and eighty-seven cents" and "pennies"—and is directly linked to physical reactions, causing cheeks to burn and prompting Della to howl. By the second block, the sacrifice has already occurred. Della does not describe the event as a loss, choosing instead to declare that her hair grows quickly, while Jim’s unreadable expression frightens her. In the final block, Jim reveals his own sale, prompting the narrator to shift from a joking tone to a direct address. The narrator labels the couple first as "foolish" and ultimately as "the wisest." This shift marks a distinct turning point, where the very acts that initially resembled poverty and waste are renamed as true wealth.

## One overall concept
The core concept uniting the marked phrases is value inversion, where the narrative language transforms a material loss into emotional value. Through this lens, sacrifice reads as richness, and apparent foolishness is reinterpreted as deep wisdom.

## My positionality
The perspective shaping this analysis is that of a college student from a busy, multilingual Indian city. While academic coursework is conducted entirely in English, it functions as an acquired register rather than the primary language spoken at home. This background naturally draws immediate attention to plot progression and emotional resonance, while rare or highly formal vocabulary is processed secondarily. Consequently, the interpretation remains highly alert to themes of financial constraint and familial obligation within a story about gift-giving, while the finer, ironic nuances of a formal English narrator may initially be less apparent.

## My research problem
Reading from the perspective of a student from an urban, multilingual Indian background, the analysis reveals that the first block treats money as a cause of shame, the second treats the body as a commodity that can be sold, and the third allows the narrator to redefine the nature of wisdom. This framework leads to the central inquiry: How does the narrator's language turn economic loss into emotional value, and what does the reader learn to call "wise"? Consequently, the core thesis to be tested is: Words of shame, counting and selling are concentrated in the first two stages of the story, while words of wisdom and the narrator's comments are concentrated in the last stage.

## My dimensions
1. **Transactional shaming.** Money handled as a source of exposure and shame, not only as a means of buying.
   (a) *amounts_money*: Sums of money, coins and wages (for example dollars, cents, pennies).
   (b) *counting_saving_bargaining*: Acts of counting, saving, calculating, bargaining and planning to buy.
   (c) *shame_and_want*: Words and phrases of shame, shabbiness, poverty and lack.
2. **Corporeal commodification.** A part of the body treated as a thing that can be priced, sold and exchanged.
   (a) *body_as_sellable*: Hair and the words used for it when it is treated as goods.
   (b) *price_sale_exchange*: Words of price, sale and exchange attached to the transaction.
   (c) *reaction_to_sale*: The body's and the self's reactions to the sale (fear, panic, self-image).
3. **Objects and feeling.** Gifts and possessions as carriers of feeling and character.
   (a) *gifts_possessions*: The gifts and possessions themselves (watch, chain, combs, presents).
   (b) *qualities_of_objects*: Qualities given to objects (beautiful, rare, precious, simple and chaste).
   (c) *feelings_attached*: Feelings that the characters attach to people and objects.
4. **Paradoxical value inversion.** The narrator turns loss into gain and foolishness into wisdom.
   (a) *wisdom_foolishness*: Words of wisdom, foolishness, reason and calculation.
   (b) *narrator_comments*: The narrator's direct addresses, asides and comments on the telling.
   (c) *ironic_register*: Mock-heroic, formal or exaggerated phrases that create irony.


# STEP 3: Text Upload

In [ ]:
import pandas as pd

text = open(STORY_PATH, encoding="utf-8").read()
text = text.replace("\r", "")

# remove obvious noise lines (page numbers, "Page x of y", copy-paste headers and footers)
noise = re.compile(r"^\s*(page\s+\d+(\s+of\s+\d+)?|\d+\s*/\s*\d+|https?://\S+|\d{1,2}/\d{1,2}/\d{2,4},?.*)\s*$", re.I)
text = "\n".join(line for line in text.split("\n") if not noise.match(line))

text = re.sub(r"[ \t]+", " ", text)
text = re.sub(r"\n{3,}", "\n\n", text).strip()
open(STORY_PATH, "w", encoding="utf-8").write(text)

paras = [re.sub(r"\s*\n\s*", " ", p).strip() for p in re.split(r"\n\s*\n", text) if p.strip()]

print(len(text.split()), "words;", len(paras), "paragraphs")
print("START:", text[:60])
print("END:  ", text[-40:])
assert len(text.split()) > 1500, "The story is missing or incomplete. Upload the file again."
assert len(paras) >= 20, "Line breaks were lost. Each paragraph must be separated by a blank line."

para_table = pd.DataFrame({"paragraph": range(1, len(paras) + 1),
                           "words": [len(p.split()) for p in paras],
                           "begins": [p[:70] for p in paras]})
para_table

2057 words; 45 paragraphs
START: One dollar and eighty-seven cents. That was all. And sixty c
END:   here they are wisest. They are the magi.


,paragraph,words,begins
0,1,67,One dollar and eighty-seven cents. That was al...
1,2,38,There was clearly nothing to do but flop down ...
2,3,48,While the mistress of the home is gradually su...
3,4,39,In the vestibule below was a letter-box into w...
4,5,80,The “Dillingham” had been flung to the breeze ...
5,6,127,Della finished her cry and attended to her che...
6,7,56,There was a pier glass between the windows of ...
7,8,40,Suddenly she whirled from the window and stood...
8,9,106,"Now, there were two possessions of the James D..."
9,10,60,So now Della’s beautiful hair fell about her r...


# STEP 4: Query Generation

In [ ]:
schema = {d: {s: [] for s in info["subs"]} for d, info in DIMENSIONS.items()}
schema_text = json.dumps(schema, indent=1, ensure_ascii=False)

def_lines = ""
for d, info in DIMENSIONS.items():
    def_lines += f"\n{info['label']} ({d}): {info['definition']}\n"
    for s, text_ in info["subs"].items():
        def_lines += f"  - {s}: {text_}\n"

PROMPT = f"""You are assisting a literary researcher who is doing a reader-oriented analysis of O. Henry's short story "The Gift of the Magi" (1905).

RESEARCH QUESTION: {RESEARCH_QUESTION}

TASK: Read the PASSAGE below. Extract the words and phrases in which the narrator's language expresses each sub-type defined here.
{def_lines}
RULES:
1. Copy every item exactly as it appears in the passage. Do not paraphrase, correct or add words.
2. Choose the shortest phrase that carries the meaning, but keep idioms and collocations whole (for example "close dealing", "silent imputation of parsimony").
3. Put each distinct phrase under ONE sub-type only. If it could fit two, choose the better fit.
4. List each distinct phrase once. Do not repeat it; the researcher's code counts the occurrences.
5. Do not extract ordinary function words or generic verbs and nouns (for example took, was, table, chops).
6. A single word is allowed only when it is central to the dimension (for example hair, sold, wisest).
7. Return ONLY valid JSON with exactly this structure, with no commentary and no markdown fences.

{schema_text}

PASSAGE:
{{{{PASSAGE}}}}
"""
open("prompt/prompt.txt", "w", encoding="utf-8").write(PROMPT)

# validation: the JSON block is valid and the placeholder is present
block = PROMPT[PROMPT.index("{\n"):PROMPT.index("PASSAGE:")]
json.loads(block)
print("JSON structure is valid. Placeholder present:", "{{PASSAGE}}" in PROMPT)

# the full prompt, ready to paste into an LLM
full_prompt = PROMPT.replace("{{PASSAGE}}", text)
open("extraction/full_prompt.txt", "w", encoding="utf-8").write(full_prompt)
print(f"\nfull_prompt.txt: {len(full_prompt.split())} words\n")
print(PROMPT)

JSON structure is valid. Placeholder present: True

full_prompt.txt: 2497 words

You are assisting a literary researcher who is doing a reader-oriented analysis of O. Henry's short story "The Gift of the Magi" (1905).

RESEARCH QUESTION: How does the narrator's language turn economic loss into emotional value, and what does the reader learn to call "wise"?

TASK: Read the PASSAGE below. Extract the words and phrases in which the narrator's language expresses each sub-type defined here.

Transactional shaming (transactional_shaming): Money handled as a source of exposure and shame, not only as a means of buying.
  - amounts_money: Sums of money, coins and wages (for example dollars, cents, pennies).
  - counting_saving_bargaining: Acts of counting, saving, calculating, bargaining and planning to buy.
  - shame_and_want: Words and phrases of shame, shabbiness, poverty and lack.

Corporeal commodification (corporeal_commodification): A part of the body treated as a thing that can be price

# STEP 5: Targeted Extraction and Formatting

In [ ]:
LLM_OUTPUT = r"""
{
  "transactional_shaming": {
    "amounts_money": [
      "one dollar and eighty-seven cents", "a dollar and eighty-seven cents", "sixty cents",
      "pennies", "penny", "$1.87", "$8", "$30", "$20", "twenty-one dollars", "87 cents",
      "twenty dollars a week"
    ],
    "counting_saving_bargaining": [
      "bulldozing", "close dealing", "counted", "saved", "saving", "calculated", "contracting",
      "expenses had been greater", "doesn't go far", "to buy jim a present",
      "to buy a present for jim"
    ],
    "shame_and_want": [
      "cheeks burned", "silent imputation of parsimony", "shabby", "mendicancy squad",
      "worn red carpet", "on the sly", "burdened with a family", "needed a new overcoat",
      "without gloves", "howl"
    ]
  },
  "corporeal_commodification": {
    "body_as_sellable": [
      "hair", "hairs of my head", "brown cascade", "tresses",
      "lifting the mass with a practised hand", "hair goods of all kinds", "cut off",
      "cut it off"
    ],
    "price_sale_exchange": [
      "will you buy my hair", "i buy hair", "twenty dollars", "sold", "exchange",
      "i sold the watch to get the money to buy your combs"
    ],
    "reaction_to_sale": [
      "panting", "give it to me quick", "a tear or two", "faltered", "nervously",
      "lost its color", "turned white", "truant schoolboy", "coney island chorus girl",
      "ravages made by generosity added to love", "please god, make him think i am still pretty",
      "my hair grows so fast", "my hair grows awfully fast", "it'll grow out again",
      "if jim doesn't kill me"
    ]
  },
  "objects_and_feeling": {
    "gifts_possessions": [
      "gold watch", "watch", "the watch", "platinum fob chain", "fob chain", "chain",
      "leather strap", "combs", "present", "presents", "gift", "gifts", "treasures",
      "possessions", "pure tortoise shell", "jewelled rims", "coveted adornments"
    ],
    "qualities_of_objects": [
      "simple and chaste in design", "quietness and value", "value", "beautiful", "rare",
      "sterling", "fine", "expensive", "worthy", "dull precious metal", "too nice", "nice",
      "grand"
    ],
    "feelings_attached": [
      "mighty pride", "envy", "worshipped", "craved", "yearned", "ecstatic scream of joy",
      "hysterical tears and wails", "terrified", "sobs, sniffles, and smiles", "hugged", "happy",
      "smile", "smiled", "ardent spirit", "intoxication", "darling", "silent prayer", "anxious",
      "serious sweetness", "dim eyes"
    ]
  },
  "paradoxical_value_inversion": {
    "wisdom_foolishness": [
      "wise", "wise men", "wonderfully wise men", "wise ones", "wisest", "foolish children",
      "unwisely", "prudence and reason", "idiocy",
      "a mathematician or a wit would give you the wrong answer", "magi"
    ],
    "narrator_comments": [
      "dear friends", "let us regard with discreet scrutiny", "perhaps you have seen",
      "take a look at the home", "as you know", "forget the hashed metaphor",
      "this dark assertion will be illuminated later on", "which is all very good",
      "they always are", "which instigates the moral reflection",
      "here i have lamely related to you",
      "in a last word to the wise of these days let it be said",
      "which is always a tremendous task", "poor fellow", "already introduced to you"
    ],
    "ironic_register": [
      "did not exactly beggar description", "tripped by on rosy wings",
      "meretricious ornamentation", "inconsequential object", "uneventful chronicle",
      "the lord of the flat", "the mistress of the home", "queen of sheba", "king solomon",
      "her majesty's jewels and gifts", "eight dollars a week or a million a year",
      "nobody could ever count my love for you", "the greatest treasures of their house",
      "the magi brought valuable gifts, but that was not among them"
    ]
  }
}
"""
open("extraction/llm_output.json", "w", encoding="utf-8").write(LLM_OUTPUT.strip())
print("Saved extraction/llm_output.json")

Saved extraction/llm_output.json


In [ ]:
MODEL_NAME = "Claude (Anthropic), claude.ai chat"      # change this if you used another model
EXTRACTION_DATE = "31 October 2026"                     # change to your own date
RESEARCHER_REJECT = []                                   # phrases you decide to cut, e.g. ["took"]
STOPLIST = ["at present"]                                # false friends: matched but never counted

raw_json = open("extraction/llm_output.json", encoding="utf-8").read().strip()
raw_json = re.sub(r"^```(?:json)?|```$", "", raw_json, flags=re.M).strip()   # remove code fences if pasted
data = json.loads(raw_json)

norm = lambda s: " ".join(str(s).lower().replace("’", "'").replace("‘", "'")
                          .replace("“", '"').replace("”", '"').split())
ntext = norm(text)
word_re = lambda k: re.compile(r"(?<!\w)" + re.escape(k) + r"(?!\w)")

# the model must use exactly the dimensions and sub-types defined in Step 2
for d, subs in data.items():
    assert d in DIMENSIONS, f"Unknown dimension: {d}"
    for s in subs:
        assert s in DIMENSIONS[d]["subs"], f"Unknown sub-type: {d} / {s}"

verified, removed, seen = {}, [], {}
for d, subs in data.items():
    for s, items in subs.items():
        for it in items:
            k = norm(it).strip(" .,;:!?\"'")
            if not k:
                continue
            if k in RESEARCHER_REJECT:
                removed.append((d, s, it, "rejected by researcher"))
            elif k in seen:
                removed.append((d, s, it, f"duplicate of {seen[k][0]} / {seen[k][1]}"))
            elif not word_re(k).search(ntext):
                removed.append((d, s, it, "not found word for word in the story"))
            else:
                seen[k] = (d, s)
                verified.setdefault(d, {}).setdefault(s, []).append(k)

json.dump(verified, open("extraction/verified_items.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
json.dump({"model": MODEL_NAME, "date": EXTRACTION_DATE, "prompt_file": "prompt/prompt.txt",
           "merged": data}, open("extraction/raw_llm_output.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)

lines = ["# Words and phrases extracted from O. Henry's \"The Gift of the Magi\" for a synthetic reading.",
         f"# Extracted by: {MODEL_NAME}, {EXTRACTION_DATE}. Verified against the text and reviewed by the researcher.",
         "", "themes = {"]
for d, subs in verified.items():
    for s, items in subs.items():
        lines.append(f'    "{d}: {s}": [')
        lines += [f"        {json.dumps(x, ensure_ascii=False)}," for x in items]
        lines.append("    ],")
lines.append("}")
open("extraction/Theme_JSON.py", "w", encoding="utf-8").write("\n".join(lines))

readme = f"""# Extraction record
- Model: {MODEL_NAME}
- Date: {EXTRACTION_DATE}
- Prompt: prompt/prompt.txt (the full version sent is extraction/full_prompt.txt)
- Output as received: extraction/llm_output.json
- Verification: every phrase must occur word for word in the story and under one sub-type only.
- Researcher rejections: {RESEARCHER_REJECT if RESEARCHER_REJECT else "none"}
- Matching in Step 6: whole words, case-insensitive, longest phrase first, no stretch of text counted twice; stoplist: {STOPLIST}
"""
open("extraction/README.md", "w", encoding="utf-8").write(readme)

n_ver = sum(len(v) for subs in verified.values() for v in subs.values())
print(f"{n_ver} phrases verified; {len(removed)} removed.\n")
for d, subs in verified.items():
    for s, items in subs.items():
        print(f"{d} / {s}: {len(items)} phrases")
print("\nREMOVED:")
for r in removed:
    print("  ", r)
if not removed:
    print("   none")

152 phrases verified; 0 removed.

transactional_shaming / amounts_money: 12 phrases
transactional_shaming / counting_saving_bargaining: 11 phrases
transactional_shaming / shame_and_want: 10 phrases
corporeal_commodification / body_as_sellable: 8 phrases
corporeal_commodification / price_sale_exchange: 6 phrases
corporeal_commodification / reaction_to_sale: 15 phrases
objects_and_feeling / gifts_possessions: 17 phrases
objects_and_feeling / qualities_of_objects: 13 phrases
objects_and_feeling / feelings_attached: 20 phrases
paradoxical_value_inversion / wisdom_foolishness: 11 phrases
paradoxical_value_inversion / narrator_comments: 15 phrases
paradoxical_value_inversion / ironic_register: 14 phrases

REMOVED:
   none


# STEP 6: Computation and Visualisation

In [ ]:
# paragraph number where each segment (scene) BEGINS
SEGMENT_STARTS = [1, 3, 6, 9, 12, 18, 20, 23, 27, 34, 37, 40, 45]
# paragraph number where each stage BEGINS (it must also be the start of a segment)
STAGES = [(1, "I. Scarcity and shame"), (9, "II. Sacrifice and exchange"),
          (23, "III. Confrontation"), (37, "IV. Revelation and revaluation")]
SEGMENT_NOTES = ["Della counts $1.87 and cries", "The flat and the Youngs' income",
                 "Della plans the present; the pier glass", "The two treasures; the hair falls; Della leaves",
                 "Madame Sofronie buys the hair", "Shopping; the platinum chain",
                 "Repairs at home; her reflection", "Waiting; Jim arrives",
                 "Della tells Jim she sold her hair", "The narrator's aside; the package",
                 "The combs are opened", "The watch; Jim's confession", "The narrator's coda: the magi"]

assert SEGMENT_STARTS[0] == 1 and SEGMENT_STARTS == sorted(set(SEGMENT_STARTS))
assert STAGES[0][0] == 1 and all(s in SEGMENT_STARTS for s, _ in STAGES)
assert SEGMENT_STARTS[-1] <= len(paras) and len(SEGMENT_NOTES) == len(SEGMENT_STARTS)

bounds = SEGMENT_STARTS + [len(paras) + 1]
segments = []
for k in range(len(SEGMENT_STARTS)):
    a, b = bounds[k], bounds[k + 1]
    seg_text = " ".join(paras[a - 1:b - 1])
    stage = [name for s, name in STAGES if s <= a][-1]
    segments.append({"id": k + 1, "paras": f"{a}-{b - 1}", "stage": stage, "note": SEGMENT_NOTES[k],
                     "text": seg_text, "words": len(seg_text.split())})

seg_of = {s["id"]: s for s in segments}
stage_names = list(dict.fromkeys(s["stage"] for s in segments))
stage_words = {st: sum(s["words"] for s in segments if s["stage"] == st) for st in stage_names}
pd.DataFrame([{k: v for k, v in s.items() if k != "text"} for s in segments])

,id,paras,stage,note,words
0,1,1-2,I. Scarcity and shame,Della counts $1.87 and cries,105
1,2,3-5,I. Scarcity and shame,The flat and the Youngs' income,167
2,3,6-8,I. Scarcity and shame,Della plans the present; the pier glass,223
3,4,9-11,II. Sacrifice and exchange,The two treasures; the hair falls; Della leaves,204
4,5,12-17,II. Sacrifice and exchange,Madame Sofronie buys the hair,80
5,6,18-19,II. Sacrifice and exchange,Shopping; the platinum chain,179
6,7,20-22,II. Sacrifice and exchange,Repairs at home; her reflection,121
7,8,23-26,III. Confrontation,Waiting; Jim arrives,210
8,9,27-33,III. Confrontation,Della tells Jim she sold her hair,207
9,10,34-36,III. Confrontation,The narrator's aside; the package,138


In [ ]:
ns = {}
exec(open("extraction/Theme_JSON.py", encoding="utf-8").read(), ns)
themes = ns["themes"]

entries = []                                   # (phrase, dimension, sub-type); stoplist entries have dimension None
for key, items in themes.items():
    d, s = [x.strip() for x in key.split(":", 1)]
    entries += [(k, d, s) for k in items]
entries += [(norm(k), None, None) for k in STOPLIST]
entries.sort(key=lambda e: -len(e[0]))         # longest phrase first

rows = []
for seg in segments:
    nt = norm(seg["text"])
    taken = [False] * len(nt)                  # characters already matched in this segment
    for k, d, s in entries:
        for m in word_re(k).finditer(nt):
            if any(taken[m.start():m.end()]):
                continue
            for i in range(m.start(), m.end()):
                taken[i] = True
            if d is not None:
                rows.append({"segment": seg["id"], "stage": seg["stage"], "dimension": d, "sub": s,
                             "item": k, "pos": m.start()})

df = pd.DataFrame(rows)
df.to_csv("extraction/matches.csv", index=False)
json.dump(rows, open("extraction/segment_counts.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)

print(len(df), "matches in", len(text.split()), "words =", round(100 * len(df) / len(text.split()), 1), "per 100 words\n")
print(df.groupby("dimension").size().rename("hits").to_string())
never = [k for k, d, s in entries if d is not None and not ((df["item"] == k).any())]
print("\nPhrases that never matched on their own (usually because a longer phrase already covers them):", never or "none")

193 matches in 2057 words = 9.4 per 100 words

dimension
corporeal_commodification      41
objects_and_feeling            72
paradoxical_value_inversion    43
transactional_shaming          37

Phrases that never matched on their own (usually because a longer phrase already covers them): none


In [ ]:
from scipy.stats import chisquare, spearmanr

DIMS = list(DIMENSIONS)
SUBS = [(d, s) for d in DIMS for s in DIMENSIONS[d]["subs"]]
total_words = sum(stage_words.values())

stage_counts = pd.crosstab(df["dimension"], df["stage"]).reindex(index=DIMS, columns=stage_names, fill_value=0)
stage_dens = stage_counts.div(pd.Series(stage_words), axis=1) * 100
print("HITS PER STAGE");            print(stage_counts.to_string())
print("\nHITS PER 100 WORDS");        print(stage_dens.round(2).to_string())

sub_counts = pd.crosstab([df["dimension"], df["sub"]], df["stage"]).reindex(columns=stage_names, fill_value=0)
print("\nSUB-TYPES PER STAGE (hits)");  print(sub_counts.to_string())

seg_counts = pd.crosstab(df["dimension"], df["segment"]).reindex(index=DIMS, columns=[s["id"] for s in segments], fill_value=0)
seg_dens = seg_counts.div(pd.Series({s["id"]: s["words"] for s in segments}), axis=1) * 100
print("\nHITS PER SEGMENT");           print(seg_counts.to_string())
print("\nHITS PER 100 WORDS PER SEGMENT"); print(seg_dens.round(1).to_string())

print("\nTESTS (chi-square across stages vs. stage length; Spearman rho of density against segment order)")
for d in DIMS:
    obs = stage_counts.loc[d].values
    exp = obs.sum() * pd.Series(stage_words)[stage_names].values / total_words
    chi = chisquare(obs, exp)
    rho = spearmanr(seg_dens.columns, seg_dens.loc[d].values)
    print(f"{DIMENSIONS[d]['label']:<30} chi2={chi.statistic:6.1f}  p={chi.pvalue:.4f}   rho={rho.statistic:+.2f}  p={rho.pvalue:.3f}")

def kwic(item, width=55, limit=8):
    """Keyword in context: the sentence fragments around a phrase, in story order."""
    k = norm(item)
    shown = 0
    for seg in segments:
        nt = norm(seg["text"])
        for m in word_re(k).finditer(nt):
            if shown >= limit:
                return
            a, b = max(0, m.start() - width), min(len(nt), m.end() + width)
            print(f"  S{seg['id']:<2} ...{nt[a:b]}...")
            shown += 1

print("\n(KWIC lists every raw occurrence, including words that sit inside a longer phrase counted elsewhere.)")
for word in ["sold", "hair", "wisest", "magi", "count"]:
    print(f"\nKWIC: {word}")
    kwic(word)

# ---- consistency check between these results and the interpretation written in Step 7 ----
EXPECTED = {"transactional_shaming": [5.86, 0.86, 0.54, 0.0], "corporeal_commodification": [0.4, 3.08, 2.88, 1.18],
            "objects_and_feeling": [1.62, 3.77, 1.98, 7.33], "paradoxical_value_inversion": [1.62, 1.54, 1.62, 4.02]}
ok = len(df) == 193 and all([round(x, 2) for x in stage_dens.loc[d].values] == EXPECTED[d] for d in DIMS)
print("\nCONSISTENCY CHECK:", "PASS - the numbers match the interpretation text in Step 7." if ok else
      "WARNING - your results differ from the numbers quoted in Step 7. Update the text in Step 7 before submitting.")

HITS PER STAGE
stage                        I. Scarcity and shame  II. Sacrifice and exchange  III. Confrontation  IV. Revelation and revaluation
dimension                                                                                                                         
transactional_shaming                           29                           5                   3                               0
corporeal_commodification                        2                          18                  16                               5
objects_and_feeling                              8                          22                  11                              31
paradoxical_value_inversion                      8                           9                   9                              17

HITS PER 100 WORDS
stage                        I. Scarcity and shame  II. Sacrifice and exchange  III. Confrontation  IV. Revelation and revaluation
dimension                                       

In [ ]:
import plotly.graph_objects as go
from collections import Counter, defaultdict
from html import escape

INK, OXBLOOD = "#1f3a5f", "#8c2f39"
PALETTE = ["#1f3a5f", "#8c2f39", "#4a7c59", "#b08d57"]
FONT = "Georgia, 'Times New Roman', serif"
CONFIG = {"displaylogo": False, "toImageButtonOptions": {"format": "png", "scale": 3, "filename": "figure"}}

pretty = lambda s: s.replace("_", " ").strip().capitalize()
dim_label = {d: f"{i + 1}. {DIMENSIONS[d]['label']}" for i, d in enumerate(DIMS)}
ids = [s["id"] for s in segments]
n = len(ids)
stage_span = [(st, min(s["id"] for s in segments if s["stage"] == st),
               max(s["id"] for s in segments if s["stage"] == st)) for st in stage_names]

by_seg_dim = defaultdict(Counter)
for r in rows:
    by_seg_dim[(r["segment"], r["dimension"])][r["item"]] += 1
top = lambda c: ", ".join(f"{k} ×{v}" for k, v in c.most_common(6)) or "none"

def style(fig, h=440, left=60):
    fig.update_layout(template="simple_white", height=h, margin=dict(l=left, r=20, t=70, b=50),
                      font=dict(family=FONT, size=13, color="#222"),
                      hoverlabel=dict(font_family=FONT, bgcolor="white", align="left"),
                      legend=dict(orientation="h", y=-0.2))
    return fig

def story_axis(fig):
    fig.update_xaxes(title="Segment (in story order)", dtick=1, range=[0.5, n + 0.5])
    for st, a, b in stage_span:
        if a != 1:
            fig.add_vline(x=a - 0.5, line_width=1.5, line_dash="dot", line_color=OXBLOOD)
        fig.add_annotation(x=(a + b) / 2, y=1.0, yref="paper", yanchor="bottom", showarrow=False,
                           text=st, font=dict(size=11, color=OXBLOOD))
    return fig

# Figure 1: heatmap, dimension x segment (colour = hits per 100 words, number = raw hits)
raw = [[sum(by_seg_dim[(i, d)].values()) for i in ids] for d in DIMS]
dens = [[100 * raw[a][b] / seg_of[i]["words"] for b, i in enumerate(ids)] for a in range(len(DIMS))]
hover = [[f"<b>{dim_label[d]}</b><br>Segment {i}: {seg_of[i]['note']}<br>{raw[a][b]} hit(s) = {dens[a][b]:.1f} per 100 words"
          f"<br>{top(by_seg_dim[(i, d)])}" for b, i in enumerate(ids)] for a, d in enumerate(DIMS)]
fig1 = go.Figure(go.Heatmap(z=dens, x=ids, y=[dim_label[d] for d in DIMS], customdata=hover,
                            text=[[str(v) for v in row] for row in raw], texttemplate="%{text}",
                            hovertemplate="%{customdata}<extra></extra>",
                            colorscale=[[0, "#f3eee4"], [1, INK]], xgap=3, ygap=3,
                            colorbar=dict(title="Hits per<br>100 words")))
style(fig1, 380, 200); fig1.update_yaxes(autorange="reversed"); story_axis(fig1)

# Figure 2: stages compared, hits per 100 words
fig2 = go.Figure()
for di, d in enumerate(DIMS):
    fig2.add_bar(x=stage_names, y=[stage_dens.loc[d, st] for st in stage_names], name=dim_label[d],
                 marker_color=PALETTE[di],
                 customdata=[[int(stage_counts.loc[d, st]), stage_words[st]] for st in stage_names],
                 hovertemplate="%{x}<br>" + dim_label[d] + "<br>%{y:.2f} per 100 words"
                               "<br>(%{customdata[0]} hits in %{customdata[1]} words)<extra></extra>")
style(fig2, 430); fig2.update_layout(barmode="group"); fig2.update_yaxes(title="Hits per 100 words")

# Figure 3: timeline, one line per dimension
fig3 = go.Figure()
for di, d in enumerate(DIMS):
    fig3.add_trace(go.Scatter(x=ids, y=[seg_dens.loc[d, i] for i in ids], mode="lines+markers", name=dim_label[d],
                              line=dict(color=PALETTE[di], width=2.5),
                              customdata=[[int(seg_counts.loc[d, i]), seg_of[i]["words"]] for i in ids],
                              hovertemplate="Segment %{x}<br>" + dim_label[d] + "<br>%{y:.2f} per 100 words"
                                            "<br>(%{customdata[0]} hits in %{customdata[1]} words)<extra></extra>"))
style(fig3, 440); fig3.update_yaxes(title="Hits per 100 words"); story_axis(fig3)

# Figure 4: the most frequent phrases
common = Counter(r["item"] for r in rows).most_common(15)
item_dim = {r["item"]: r["dimension"] for r in rows}
fig4 = go.Figure(go.Bar(x=[v for _, v in common][::-1], y=[k for k, _ in common][::-1], orientation="h",
                        marker_color=[PALETTE[DIMS.index(item_dim[k])] for k, _ in common][::-1],
                        hovertemplate="%{y}: %{x} hit(s)<extra></extra>"))
style(fig4, 460, 230); fig4.update_xaxes(title="Number of occurrences")
fig4.update_layout(showlegend=False)

FIGS = [fig1, fig2, fig3, fig4]
for f in FIGS:
    f.show(config=CONFIG)

# STEP 7: Interpretation of the Result

In [ ]:
INTERPRETATION = {
    1: "Shame vocabulary is densest in segment 1 (12.4 hits per 100 words) and is absent from segment 9 onwards; corporeal vocabulary peaks at the sale (segment 5, 10.0) and again at the confession (segment 9, 6.8); objects and feeling peak when the combs are opened (segment 11, 12.0); inversion peaks only in the final paragraph (segment 13, 13.1).",
    2: "Shaming falls from 5.86 to 0.00 hits per 100 words across the four stages, corporeal commodification peaks in Stages II and III (3.08 and 2.88), and objects and feeling and inversion both peak in Stage IV (7.33 and 4.02).",
    3: "Only shaming declines steadily (Spearman rho = -0.81); the other three dimensions move in bursts tied to scenes: the sale, the confession, the opening of the combs and the final paragraph.",
    4: "\"hair\" is the most frequent single item (9), followed by \"beautiful\" (5) and \"sold\", \"combs\" and \"gifts\" (4 each): the vocabulary of value is concentrated in a few words about the body and the gifts.",
}
HOW = {
    1: "Each row is one dimension and each column is one segment of the story, in order. The number in each square is the number of hits; the shade shows hits per 100 words, so short and long segments can be compared. Dotted red lines mark where a new stage begins.",
    2: "The story is grouped into four stages. Each bar shows hits per 100 words, so a longer stage does not look stronger only because it is longer.",
    3: "Each line is one dimension followed across the segments in story order, in hits per 100 words. A rising line means that the dimension grows denser as the story moves on.",
    4: "The fifteen most frequent phrases in the extraction, coloured by dimension.",
}
TITLES = {1: "Figure 1. Where does each dimension appear in the story?",
          2: "Figure 2. How do the stages differ?",
          3: "Figure 3. How does each dimension change across the story?",
          4: "Figure 4. Which phrases carry the most weight?"}

In [ ]:
ESSAY = r"""
# Interpretation: how the narrator's language turns economic loss into emotional value in "The Gift of the Magi"

## 7.1 Basis of the interpretation

This interpretation rests on the results of Step 6: the table of 193 matched phrases, the counts per stage and segment, the two statistical tests, the keyword-in-context lines, and the four figures. It follows the reader-oriented approach of the project: the four dimensions came from the initial reading of the story, and the counts are used to check, refine and sometimes correct that first impression.

The counts are descriptive. They show where each kind of language appears in the story and how often. They do not show tone or meaning on their own, so every pattern below is read together with the passage of the story in which it occurs.

Throughout, results are given as hits per 100 words, so that stages and segments of different lengths can be compared fairly. Stage I has 495 words, Stage II 584, Stage III 555 and Stage IV 423. The segments range from 80 to 223 words.

## 7.2 The overall picture

The extraction found 193 phrase occurrences in 2,057 words, which is 9.4 hits per 100 words. Each occurrence belongs to exactly one sub-type, so nothing is counted twice. By dimension the totals are:

- Objects and feeling: 72 hits (37% of all hits)
- Paradoxical value inversion: 43 hits
- Corporeal commodification: 41 hits
- Transactional shaming: 37 hits

The vocabulary of objects and feeling is therefore the largest single field in the story. The other three dimensions are of similar size, but, as the figures show, they are located in very different places.

## 7.3 Figure 1: where each dimension appears (heatmap)

The heatmap shows each dimension in each of the 13 segments. The number in each square is the count of hits, and the shade shows hits per 100 words.

**Transactional shaming (row 1)** is dark only at the start. Segment 1 (Della counts her money) has 13 hits in 105 words, which is 12.4 per 100 words. Segment 2 (the flat and the income) has 5 hits and segment 3 (Della plans the present) has 11 hits. After that the row fades: 1, 0, 3, 1 and 3 hits in segments 4 to 8, and then **zero in every segment from 9 to 13**. Shame and money language belongs to the opening of the story and then disappears.

**Corporeal commodification (row 2)** has two clear peaks:
- Segment 5, the sale to Madame Sofronie, has 8 hits in only 80 words (10.0 per 100 words). Every hit in this segment belongs to this one dimension, and there are no hits at all for shame, objects or inversion. The scene is entirely about the transaction.
- Segment 9, where Della tells Jim what she has done, has 14 hits (6.8 per 100 words), the highest count of any segment for this dimension.

Between and around the peaks, the row is almost empty: 0 hits in segments 1 and 2, and 0 in segments 6 and 10.

**Objects and feeling (row 3)** is the darkest row overall and has two main peaks: segment 6, the search for the platinum chain (14 hits, 7.8 per 100 words), and segment 11, the opening of the combs (18 hits in 150 words, 12.0 per 100 words, the highest density of any square in this row). Smaller concentrations occur in segment 4 (7 hits, the two treasures), segment 9 (8 hits), segment 12 (8 hits, Jim's confession about the watch) and segment 13 (5 hits). The row is empty in segment 5 (the sale) and segment 10.

**Paradoxical value inversion (row 4)** is light in almost every square and then very dark at the end: segment 13, the narrator's closing paragraph, has 16 hits in 122 words, which is 13.1 per 100 words. The next highest squares are segment 10 (6 hits, 4.3 per 100 words) and segment 2 (5 hits, 3.0 per 100 words).

**The busiest segment.** The final paragraph (segment 13) has 22 hits in 122 words, which is 18.0 per 100 words, the highest overall density of any segment. Segment 11 (14.7 per 100 words) and segment 1 (14.3 per 100 words) follow. The story is therefore most densely marked at the start (money), at the opening of the combs (objects and feeling) and at the end (the narrator's verdict).

## 7.4 Figure 2: how the four stages differ (grouped bars)

The stage chart combines the segments into four stages and shows hits per 100 words.

| Dimension | Stage I | Stage II | Stage III | Stage IV |
|---|---|---|---|---|
| Transactional shaming | 5.86 | 0.86 | 0.54 | 0.00 |
| Corporeal commodification | 0.40 | 3.08 | 2.88 | 1.18 |
| Objects and feeling | 1.62 | 3.77 | 1.98 | 7.33 |
| Paradoxical value inversion | 1.62 | 1.54 | 1.62 | 4.02 |

**Shaming falls in every stage.** It drops from 5.86 in Stage I to 0.86, 0.54 and finally 0.00. Stage I alone holds 29 of its 37 hits.

**Corporeal commodification rises and then falls.** It is almost absent in Stage I (0.40), reaches its highest level in Stage II (3.08), stays nearly as high in Stage III (2.88) and falls to 1.18 in Stage IV. The sale (Stage II) and the confession (Stage III) are the two high stages.

**Objects and feeling is the strongest in Stage IV**, at 7.33, which is almost twice the next highest stage (3.77 in Stage II). In Stage III it drops to 1.98, so the language of objects is not a steady rise. It is strongest during the shopping and in the final scenes.

**Value inversion is flat and then jumps.** It stays between 1.5 and 1.6 for Stages I to III and rises to 4.02 in Stage IV. Within Stage IV, 16 of the 17 hits come from the last paragraph.

**Statistical check.** A chi-square test compares the actual distribution across stages with an even spread (the spread expected from stage length alone, 3 degrees of freedom). All four dimensions differ clearly from an even spread: shaming (chi-square 60.7, p < .001), corporeal commodification (13.3, p = .004), objects and feeling (26.5, p < .001) and inversion (9.5, p = .023). None of the four patterns is likely to be an accident of where the words happen to fall. Because neighbouring words influence each other, the p-values are a rough guide and not formal proof.

## 7.5 Figure 3: change across the story (timeline)

The timeline follows each dimension through the 13 segments in order.

**Shaming is the only dimension with a steady trend.** Its line starts high and falls. The correlation between segment order and density is strongly negative (rho = -0.81, p < .001).

**The other three dimensions move in bursts, not trends.** Their correlations are small and not significant (corporeal 0.13, objects and feeling 0.46, inversion 0.04). Each line rises sharply at particular scenes and drops in between:
- Corporeal commodification spikes at the sale (segment 5) and the confession (segment 9).
- Objects and feeling spikes at the chain (segment 6) and the combs (segment 11). Between them, in segment 7 (Della's repairs at home), it falls to 1 hit (0.8 per 100 words), so the language of objects rises and falls with the scenes in which gifts are handled.
- Inversion stays low until it rises abruptly in the last point of the line.

The timeline therefore shows that only the language of shame changes gradually. The other three change scene by scene.

## 7.6 Figure 4: the most frequent phrases

The ranking of the fifteen most frequent phrases shows which words carry the weight of the analysis.

- **"hair" is the most frequent single item, with 9 occurrences.** It appears once in segment 3 (Della's plan), three times in segment 4 (the hair falls), four times in segment 9 (the confession) and once in segment 11 (the combs). The word is mentioned before, during and after the sale.
- **"beautiful" follows with 5 occurrences**, and **"sold", "combs" and "gifts"** have 4 each. All four occurrences of "sold" are in segment 9, in Della's confession. Jim's own sale is stated once, as a complete sentence ("I sold the watch to get the money to buy your combs") in segment 12.
- The ranking is dominated by words about the body and about the gifts. Words of money and counting appear in the list less often, because they were spread over many different phrases ("pennies", "$8", "saved", "counted") and each of them occurs only a few times.

## 7.7 Interpretation by dimension

### Transactional shaming: how counting becomes shame

Within this dimension, the three sub-types are not equal. Words for amounts of money have 16 hits (13 in Stage I and 3 in Stage II), counting, saving and bargaining has 11 hits (all 11 in Stage I) and words of shame and want have 10 hits (5, 2 and 3 in Stages I to III).

The opening paragraph shows the pattern. The sum "one dollar and eighty-seven cents" is stated twice in 67 words, "pennies" appears twice, and the narrator describes how the pennies were saved "by bulldozing the grocer and the vegetable man and the butcher" until "one's cheeks burned with the silent imputation of parsimony". The shame here is not caused by poverty alone. It is caused by being watched while bargaining: the grocer, the vegetable man and the butcher are the observers.

The vocabulary of shame and want continues into the description of the home (segment 2, "mendicancy squad"), and returns in smaller form when Jim "sometimes looked at it on the sly" at his watch because of its old strap, and when the narrator mentions that he "needed a new overcoat and he was without gloves". These are all marks of lack that are felt as embarrassment.

Counting vocabulary is completely absent after Stage I. The only later mention of counting is a negation, in Della's words to Jim: "nobody could ever count my love for you". The narrator's remark in segment 10, "Eight dollars a week or a million a year—what is the difference?", also removes the importance of quantity. The story starts in numbers and moves away from them, which is the first step in turning economic loss into emotional value.

### Corporeal commodification: the sale of the hair

This dimension is mainly about hair. The three sub-types differ in timing:
- **Hair as goods (17 hits)** is spread across Stages II and III (6 and 8 hits) and is the main content of the dimension.
- **Price, sale and exchange (9 hits)** is concentrated in Stage III (4 hits) and Stage II (3 hits), with the remainder in Stage IV.
- **Reactions to the sale (15 hits)** are mostly in Stage II (9 hits, 1.54 per 100 words): "panting", "a tear or two", "faltered", "nervously", "truant schoolboy". They describe Della's body and her appearance after the sale.

The sale scene (segment 5) is the most concentrated passage in the story. In 80 words it contains the sign "Hair Goods of All Kinds", the question "Will you buy my hair?", Madame's "I buy hair" and "lifting the mass with a practised hand", the price "Twenty dollars", and Della's "Give it to me quick". The hair is turned from a part of Della into goods with a price within a few lines.

The second peak (segment 9) is a different use of the same vocabulary. Della uses the words of sale as a confession and an appeal to Jim: "I had my hair cut off and sold", "Cut it off and sold it", "It's sold, I tell you—sold and gone, too". This is why the corporeal dimension remains high in Stage III (2.88 per 100 words) and why the original expectation that selling language belongs to the first two stages is only partly correct.

**An unequal distribution of the language of sacrifice.** The data show that the two sacrifices are not described in the same way. Della's sacrifice takes up two segments with high density (segments 5 and 9), nine reaction words and all four occurrences of "sold". Jim's sacrifice, the sale of the watch, is reported in one sentence in segment 12, and there are no words of bodily reaction attached to it. The watch is also described as an inherited object, "his father's and his grandfather's". The language of the story spends far more space on the sale of the body than on the sale of the object. This is an observation about the distribution of words, not a statement of what the story intends: the last paragraph calls both gifts "the greatest treasures of their house".

### Objects and feeling: where value is attached

This is the largest dimension, with 72 hits (3.50 per 100 words). Its three sub-types are gifts and possessions (30 hits), feelings attached (22 hits) and qualities of objects (20 hits).

**Feelings grow steadily.** Hits for feelings per 100 words are 0.61, 0.68, 0.90 and 2.36 in Stages I to IV. This is the only sub-type among the four dimensions that rises in every stage.

**Gifts and possessions** are absent in Stage I (0 hits), appear in Stage II (2.05 per 100 words), fall in Stage III (0.54) and reach their highest level in Stage IV (3.55).

**Segment 6, the chain.** In this passage (14 hits) the platinum fob chain is described with qualities that Della also attributes to Jim: "simple and chaste in design", "Quietness and value—the description applied to both". The value of the chain is explained by its likeness to Jim, so the object is valued for the person.

**Segment 11, the combs.** This segment has the highest density of this dimension (12.0 per 100 words) and includes 8 words of feeling: "worshipped", "craved", "yearned", "ecstatic scream of joy", "hysterical tears and wails". The combs are described with the language of beauty and expense ("beautiful", "pure tortoise shell", "jewelled rims", "expensive", "coveted adornments") at the exact moment when they cannot be used, because the hair they were bought for has been sold.

This is the main finding of the dimension: the strongest language of value and desire is attached to the combs when they have lost their practical use. The loss does not reduce the value of the gift in the language of the story. It increases it.

### Paradoxical value inversion: the narrator's voice

The three sub-types behave differently.

**Narrator comments (15 hits)** are not concentrated at the end. Stage I has 6 hits (1.21 per 100 words), the other stages 3 each. The final paragraph has 3 hits in 122 words (about 2.5 per 100 words), which is the highest density of any single segment. The narrator is therefore present at the start and at the end of the story, with a lower level in between. The comments also have different functions. Early comments, such as "Which is all very good" and the phrase "did not exactly beggar description", appear in the same segment as the shame vocabulary (segment 2) and make the poverty lighter in tone. The comments in the middle of the story ("Forget the hashed metaphor", "dear friends") speak directly to the reader. The comments at the end give the narrator's final verdict.

**Wisdom and foolishness (14 hits)** are almost entirely at the end: 11 of the 14 hits are in segment 13 ("wise men", "wonderfully wise men", "foolish children", "unwisely", "wisest", "magi"). The word "wise" does not occur before the last paragraph. Before it, the vocabulary of reason is used for moments of calculation and blankness: "prudence and reason" when Della calms down after shopping (segment 7), "an air almost of idiocy" for Jim after hearing about the hair (segment 9) and "A mathematician or a wit would give you the wrong answer" in the narrator's aside (segment 10). The reader is therefore first shown that calculation is not the right way to judge what has happened, and only in the last paragraph is told who is wise.

**Ironic register (14 hits)** is thin in Stage I (0.40 per 100 words) and stronger in Stages II to IV (0.86, 0.72 and 0.71). An example is segment 4, where the narrator compares Della's hair and Jim's watch with the jewels of the queen of Sheba and the treasures of King Solomon. The hair and the watch are raised above royal treasure through exaggeration, and this prepares the valuation stated openly at the end.

## 7.8 Testing the original claim

The original claim was: *words of shame, counting and selling are concentrated in the first two stages, while words of wisdom and the narrator's comments are concentrated in the last stage.*

| Part of the claim | Result | Evidence |
|---|---|---|
| Shame and counting words are early | **Supported, more strongly than claimed** | 29 of 37 shame hits and all 11 counting hits are in Stage I alone; shame falls steadily (rho = -0.81) |
| Selling words are in the first two stages | **Partly supported** | Selling and body vocabulary is highest in Stages II and III (3.08 and 2.88 per 100 words) and returns in Della's confession (segment 9) |
| Wisdom words are in the last stage | **Supported, with a qualification** | 11 of 14 hits are in a single paragraph (segment 13) |
| Narrator comments are in the last stage | **Not supported as stated** | They are highest in Stage I (6 hits) and again in the final paragraph (3 hits); the narrator is present at both ends |

## 7.9 Answer to the research question and return to the initial impression

The research question was: *How does the narrator's language turn economic loss into emotional value, and what does the reader learn to call "wise"?*

The initial impression, formed after the partial reading, was that money and love pull against each other and that the narrator's language turns loss into value. The results support this impression and show how it works. The conversion happens in four overlapping moves:

1. **Counting and shame come first.** The story opens in money (Stage I: 5.86 shame hits per 100 words) and in the embarrassment of scarcity.
2. **Numbers disappear.** Counting words are absent after Stage I and amounts of money are absent after Stage II.
3. **The sale of the hair is told in the language of goods and then in the language of confession.** The body is priced in segment 5 and the sale is spoken of again in segment 9.
4. **Feeling and value move onto the gifts.** The words of beauty, desire and joy are strongest around the combs (segment 11) and in the final stage (7.33 per 100 words), at the moment the gifts cannot be used.

**What the reader learns to call "wise".** The narrator names the result only in the last paragraph, where 11 of the 14 wisdom words appear. The reader is led to feel the value of the gifts before being told to call it wisdom, and the earlier comments about calculation prepare the final reversal: the two characters who gave up their most valuable possessions are called "the wisest".

The results also correct the first impression in two ways. The conversion is not one smooth process but several, running at different speeds. And the sale of the body is as prominent in the language as money, and returns in Stage III after the money language has gone.

## 7.10 Position of the reader and role of the model

The story was read by a college student from a busy, multilingual urban Indian setting, with English as an academic language rather than the language of the home. This position shaped what was noticed first: money, family obligation and feeling. Several of the most telling shame words, such as "parsimony", "mendicancy" and "meretricious", belong to a formal literary register that is easy to pass over when reading for plot and feeling. The extraction step made them visible, so the model complemented the reading rather than replacing it. The choice of the four dimensions, however, came from the reading itself. Another reader might begin from different concerns and arrive at a different set of dimensions.

## 7.11 Limitations

- **The method depends on a word list.** Words such as "present", "nice" and "hair" can have several senses, and counting treats all senses alike. One false positive ("just at present", meaning "now", not "gift") was caught with a stoplist, but others may remain.
- **The extraction was done by a language model** from a prompt written for this study. Every phrase was checked against the text, but the choice of which phrases to extract could differ for another model or another reader.
- **The division into 13 segments and 4 stages** was decided by the researcher from changes of scene. A different division would change the stage figures, although the strongest patterns (the fall of shaming and the final peak of inversion) are also visible in the individual segments.
- **The story is short.** A short segment, such as segment 5 with 80 words, can show a high density from a few hits.
- **Counts show vocabulary, not tone.** Irony in particular is only partly captured by phrases.
- **The statistical tests are a guide only**, because hits within a passage are not independent.

## 7.12 Conclusion

**Answer to the research question.** The narrator's language turns economic loss into emotional value gradually, in four steps, and teaches the reader to call the two characters "wise" only at the very end.

1. **Counting and shame come first.** The story opens in money: shame vocabulary is at 5.86 hits per 100 words in Stage I and falls steadily to 0.00 by Stage IV.
2. **The numbers disappear.** Counting words are absent after Stage I.
3. **The hair is sold in the language of goods and then of confession.** The sale (segment 5) and Della's confession (segment 9) are the two peaks of the corporeal vocabulary.
4. **Feeling and value move onto the gifts.** Words of beauty, desire and joy are strongest around the combs (segment 11) and in Stage IV (7.33 per 100 words), at the moment the gifts cannot be used.

**What the reader learns to call "wise".** The narrator names the result in the last paragraph, where 11 of the 14 wisdom words appear: the couple who sacrificed their most valuable possessions for each other are "the wisest" and "the magi". Earlier comments about calculation ("A mathematician or a wit would give you the wrong answer") prepare this reversal.

**Two additional findings.** The original claim is only partly supported: shame and counting are early, as expected, but selling language returns in Stage III and the narrator's comments appear at both the start and the end. The language of sacrifice is also unequal: most of the vocabulary of selling and of bodily reaction belongs to Della.
"""
ESSAY = ESSAY.strip()
open("docs/interpretation.md", "w", encoding="utf-8").write(ESSAY)
display(Markdown(ESSAY))

# Interpretation: how the narrator's language turns economic loss into emotional value in "The Gift of the Magi"

## 7.1 Basis of the interpretation

This interpretation rests on the results of Step 6: the table of 193 matched phrases, the counts per stage and segment, the two statistical tests, the keyword-in-context lines, and the four figures. It follows the reader-oriented approach of the project: the four dimensions came from the initial reading of the story, and the counts are used to check, refine and sometimes correct that first impression.

The counts are descriptive. They show where each kind of language appears in the story and how often. They do not show tone or meaning on their own, so every pattern below is read together with the passage of the story in which it occurs.

Throughout, results are given as hits per 100 words, so that stages and segments of different lengths can be compared fairly. Stage I has 495 words, Stage II 584, Stage III 555 and Stage IV 423. The segments range from 80 to 223 words.

## 7.2 The overall picture

The extraction found 193 phrase occurrences in 2,057 words, which is 9.4 hits per 100 words. Each occurrence belongs to exactly one sub-type, so nothing is counted twice. By dimension the totals are:

- Objects and feeling: 72 hits (37% of all hits)
- Paradoxical value inversion: 43 hits
- Corporeal commodification: 41 hits
- Transactional shaming: 37 hits

The vocabulary of objects and feeling is therefore the largest single field in the story. The other three dimensions are of similar size, but, as the figures show, they are located in very different places.

## 7.3 Figure 1: where each dimension appears (heatmap)

The heatmap shows each dimension in each of the 13 segments. The number in each square is the count of hits, and the shade shows hits per 100 words.

**Transactional shaming (row 1)** is dark only at the start. Segment 1 (Della counts her money) has 13 hits in 105 words, which is 12.4 per 100 words. Segment 2 (the flat and the income) has 5 hits and segment 3 (Della plans the present) has 11 hits. After that the row fades: 1, 0, 3, 1 and 3 hits in segments 4 to 8, and then **zero in every segment from 9 to 13**. Shame and money language belongs to the opening of the story and then disappears.

**Corporeal commodification (row 2)** has two clear peaks:
- Segment 5, the sale to Madame Sofronie, has 8 hits in only 80 words (10.0 per 100 words). Every hit in this segment belongs to this one dimension, and there are no hits at all for shame, objects or inversion. The scene is entirely about the transaction.
- Segment 9, where Della tells Jim what she has done, has 14 hits (6.8 per 100 words), the highest count of any segment for this dimension.

Between and around the peaks, the row is almost empty: 0 hits in segments 1 and 2, and 0 in segments 6 and 10.

**Objects and feeling (row 3)** is the darkest row overall and has two main peaks: segment 6, the search for the platinum chain (14 hits, 7.8 per 100 words), and segment 11, the opening of the combs (18 hits in 150 words, 12.0 per 100 words, the highest density of any square in this row). Smaller concentrations occur in segment 4 (7 hits, the two treasures), segment 9 (8 hits), segment 12 (8 hits, Jim's confession about the watch) and segment 13 (5 hits). The row is empty in segment 5 (the sale) and segment 10.

**Paradoxical value inversion (row 4)** is light in almost every square and then very dark at the end: segment 13, the narrator's closing paragraph, has 16 hits in 122 words, which is 13.1 per 100 words. The next highest squares are segment 10 (6 hits, 4.3 per 100 words) and segment 2 (5 hits, 3.0 per 100 words).

**The busiest segment.** The final paragraph (segment 13) has 22 hits in 122 words, which is 18.0 per 100 words, the highest overall density of any segment. Segment 11 (14.7 per 100 words) and segment 1 (14.3 per 100 words) follow. The story is therefore most densely marked at the start (money), at the opening of the combs (objects and feeling) and at the end (the narrator's verdict).

## 7.4 Figure 2: how the four stages differ (grouped bars)

The stage chart combines the segments into four stages and shows hits per 100 words.

| Dimension | Stage I | Stage II | Stage III | Stage IV |
|---|---|---|---|---|
| Transactional shaming | 5.86 | 0.86 | 0.54 | 0.00 |
| Corporeal commodification | 0.40 | 3.08 | 2.88 | 1.18 |
| Objects and feeling | 1.62 | 3.77 | 1.98 | 7.33 |
| Paradoxical value inversion | 1.62 | 1.54 | 1.62 | 4.02 |

**Shaming falls in every stage.** It drops from 5.86 in Stage I to 0.86, 0.54 and finally 0.00. Stage I alone holds 29 of its 37 hits.

**Corporeal commodification rises and then falls.** It is almost absent in Stage I (0.40), reaches its highest level in Stage II (3.08), stays nearly as high in Stage III (2.88) and falls to 1.18 in Stage IV. The sale (Stage II) and the confession (Stage III) are the two high stages.

**Objects and feeling is the strongest in Stage IV**, at 7.33, which is almost twice the next highest stage (3.77 in Stage II). In Stage III it drops to 1.98, so the language of objects is not a steady rise. It is strongest during the shopping and in the final scenes.

**Value inversion is flat and then jumps.** It stays between 1.5 and 1.6 for Stages I to III and rises to 4.02 in Stage IV. Within Stage IV, 16 of the 17 hits come from the last paragraph.

**Statistical check.** A chi-square test compares the actual distribution across stages with an even spread (the spread expected from stage length alone, 3 degrees of freedom). All four dimensions differ clearly from an even spread: shaming (chi-square 60.7, p < .001), corporeal commodification (13.3, p = .004), objects and feeling (26.5, p < .001) and inversion (9.5, p = .023). None of the four patterns is likely to be an accident of where the words happen to fall. Because neighbouring words influence each other, the p-values are a rough guide and not formal proof.

## 7.5 Figure 3: change across the story (timeline)

The timeline follows each dimension through the 13 segments in order.

**Shaming is the only dimension with a steady trend.** Its line starts high and falls. The correlation between segment order and density is strongly negative (rho = -0.81, p < .001).

**The other three dimensions move in bursts, not trends.** Their correlations are small and not significant (corporeal 0.13, objects and feeling 0.46, inversion 0.04). Each line rises sharply at particular scenes and drops in between:
- Corporeal commodification spikes at the sale (segment 5) and the confession (segment 9).
- Objects and feeling spikes at the chain (segment 6) and the combs (segment 11). Between them, in segment 7 (Della's repairs at home), it falls to 1 hit (0.8 per 100 words), so the language of objects rises and falls with the scenes in which gifts are handled.
- Inversion stays low until it rises abruptly in the last point of the line.

The timeline therefore shows that only the language of shame changes gradually. The other three change scene by scene.

## 7.6 Figure 4: the most frequent phrases

The ranking of the fifteen most frequent phrases shows which words carry the weight of the analysis.

- **"hair" is the most frequent single item, with 9 occurrences.** It appears once in segment 3 (Della's plan), three times in segment 4 (the hair falls), four times in segment 9 (the confession) and once in segment 11 (the combs). The word is mentioned before, during and after the sale.
- **"beautiful" follows with 5 occurrences**, and **"sold", "combs" and "gifts"** have 4 each. All four occurrences of "sold" are in segment 9, in Della's confession. Jim's own sale is stated once, as a complete sentence ("I sold the watch to get the money to buy your combs") in segment 12.
- The ranking is dominated by words about the body and about the gifts. Words of money and counting appear in the list less often, because they were spread over many different phrases ("pennies", "$8", "saved", "counted") and each of them occurs only a few times.

## 7.7 Interpretation by dimension

### Transactional shaming: how counting becomes shame

Within this dimension, the three sub-types are not equal. Words for amounts of money have 16 hits (13 in Stage I and 3 in Stage II), counting, saving and bargaining has 11 hits (all 11 in Stage I) and words of shame and want have 10 hits (5, 2 and 3 in Stages I to III).

The opening paragraph shows the pattern. The sum "one dollar and eighty-seven cents" is stated twice in 67 words, "pennies" appears twice, and the narrator describes how the pennies were saved "by bulldozing the grocer and the vegetable man and the butcher" until "one's cheeks burned with the silent imputation of parsimony". The shame here is not caused by poverty alone. It is caused by being watched while bargaining: the grocer, the vegetable man and the butcher are the observers.

The vocabulary of shame and want continues into the description of the home (segment 2, "mendicancy squad"), and returns in smaller form when Jim "sometimes looked at it on the sly" at his watch because of its old strap, and when the narrator mentions that he "needed a new overcoat and he was without gloves". These are all marks of lack that are felt as embarrassment.

Counting vocabulary is completely absent after Stage I. The only later mention of counting is a negation, in Della's words to Jim: "nobody could ever count my love for you". The narrator's remark in segment 10, "Eight dollars a week or a million a year—what is the difference?", also removes the importance of quantity. The story starts in numbers and moves away from them, which is the first step in turning economic loss into emotional value.

### Corporeal commodification: the sale of the hair

This dimension is mainly about hair. The three sub-types differ in timing:
- **Hair as goods (17 hits)** is spread across Stages II and III (6 and 8 hits) and is the main content of the dimension.
- **Price, sale and exchange (9 hits)** is concentrated in Stage III (4 hits) and Stage II (3 hits), with the remainder in Stage IV.
- **Reactions to the sale (15 hits)** are mostly in Stage II (9 hits, 1.54 per 100 words): "panting", "a tear or two", "faltered", "nervously", "truant schoolboy". They describe Della's body and her appearance after the sale.

The sale scene (segment 5) is the most concentrated passage in the story. In 80 words it contains the sign "Hair Goods of All Kinds", the question "Will you buy my hair?", Madame's "I buy hair" and "lifting the mass with a practised hand", the price "Twenty dollars", and Della's "Give it to me quick". The hair is turned from a part of Della into goods with a price within a few lines.

The second peak (segment 9) is a different use of the same vocabulary. Della uses the words of sale as a confession and an appeal to Jim: "I had my hair cut off and sold", "Cut it off and sold it", "It's sold, I tell you—sold and gone, too". This is why the corporeal dimension remains high in Stage III (2.88 per 100 words) and why the original expectation that selling language belongs to the first two stages is only partly correct.

**An unequal distribution of the language of sacrifice.** The data show that the two sacrifices are not described in the same way. Della's sacrifice takes up two segments with high density (segments 5 and 9), nine reaction words and all four occurrences of "sold". Jim's sacrifice, the sale of the watch, is reported in one sentence in segment 12, and there are no words of bodily reaction attached to it. The watch is also described as an inherited object, "his father's and his grandfather's". The language of the story spends far more space on the sale of the body than on the sale of the object. This is an observation about the distribution of words, not a statement of what the story intends: the last paragraph calls both gifts "the greatest treasures of their house".

### Objects and feeling: where value is attached

This is the largest dimension, with 72 hits (3.50 per 100 words). Its three sub-types are gifts and possessions (30 hits), feelings attached (22 hits) and qualities of objects (20 hits).

**Feelings grow steadily.** Hits for feelings per 100 words are 0.61, 0.68, 0.90 and 2.36 in Stages I to IV. This is the only sub-type among the four dimensions that rises in every stage.

**Gifts and possessions** are absent in Stage I (0 hits), appear in Stage II (2.05 per 100 words), fall in Stage III (0.54) and reach their highest level in Stage IV (3.55).

**Segment 6, the chain.** In this passage (14 hits) the platinum fob chain is described with qualities that Della also attributes to Jim: "simple and chaste in design", "Quietness and value—the description applied to both". The value of the chain is explained by its likeness to Jim, so the object is valued for the person.

**Segment 11, the combs.** This segment has the highest density of this dimension (12.0 per 100 words) and includes 8 words of feeling: "worshipped", "craved", "yearned", "ecstatic scream of joy", "hysterical tears and wails". The combs are described with the language of beauty and expense ("beautiful", "pure tortoise shell", "jewelled rims", "expensive", "coveted adornments") at the exact moment when they cannot be used, because the hair they were bought for has been sold.

This is the main finding of the dimension: the strongest language of value and desire is attached to the combs when they have lost their practical use. The loss does not reduce the value of the gift in the language of the story. It increases it.

### Paradoxical value inversion: the narrator's voice

The three sub-types behave differently.

**Narrator comments (15 hits)** are not concentrated at the end. Stage I has 6 hits (1.21 per 100 words), the other stages 3 each. The final paragraph has 3 hits in 122 words (about 2.5 per 100 words), which is the highest density of any single segment. The narrator is therefore present at the start and at the end of the story, with a lower level in between. The comments also have different functions. Early comments, such as "Which is all very good" and the phrase "did not exactly beggar description", appear in the same segment as the shame vocabulary (segment 2) and make the poverty lighter in tone. The comments in the middle of the story ("Forget the hashed metaphor", "dear friends") speak directly to the reader. The comments at the end give the narrator's final verdict.

**Wisdom and foolishness (14 hits)** are almost entirely at the end: 11 of the 14 hits are in segment 13 ("wise men", "wonderfully wise men", "foolish children", "unwisely", "wisest", "magi"). The word "wise" does not occur before the last paragraph. Before it, the vocabulary of reason is used for moments of calculation and blankness: "prudence and reason" when Della calms down after shopping (segment 7), "an air almost of idiocy" for Jim after hearing about the hair (segment 9) and "A mathematician or a wit would give you the wrong answer" in the narrator's aside (segment 10). The reader is therefore first shown that calculation is not the right way to judge what has happened, and only in the last paragraph is told who is wise.

**Ironic register (14 hits)** is thin in Stage I (0.40 per 100 words) and stronger in Stages II to IV (0.86, 0.72 and 0.71). An example is segment 4, where the narrator compares Della's hair and Jim's watch with the jewels of the queen of Sheba and the treasures of King Solomon. The hair and the watch are raised above royal treasure through exaggeration, and this prepares the valuation stated openly at the end.

## 7.8 Testing the original claim

The original claim was: *words of shame, counting and selling are concentrated in the first two stages, while words of wisdom and the narrator's comments are concentrated in the last stage.*

| Part of the claim | Result | Evidence |
|---|---|---|
| Shame and counting words are early | **Supported, more strongly than claimed** | 29 of 37 shame hits and all 11 counting hits are in Stage I alone; shame falls steadily (rho = -0.81) |
| Selling words are in the first two stages | **Partly supported** | Selling and body vocabulary is highest in Stages II and III (3.08 and 2.88 per 100 words) and returns in Della's confession (segment 9) |
| Wisdom words are in the last stage | **Supported, with a qualification** | 11 of 14 hits are in a single paragraph (segment 13) |
| Narrator comments are in the last stage | **Not supported as stated** | They are highest in Stage I (6 hits) and again in the final paragraph (3 hits); the narrator is present at both ends |

## 7.9 Answer to the research question and return to the initial impression

The research question was: *How does the narrator's language turn economic loss into emotional value, and what does the reader learn to call "wise"?*

The initial impression, formed after the partial reading, was that money and love pull against each other and that the narrator's language turns loss into value. The results support this impression and show how it works. The conversion happens in four overlapping moves:

1. **Counting and shame come first.** The story opens in money (Stage I: 5.86 shame hits per 100 words) and in the embarrassment of scarcity.
2. **Numbers disappear.** Counting words are absent after Stage I and amounts of money are absent after Stage II.
3. **The sale of the hair is told in the language of goods and then in the language of confession.** The body is priced in segment 5 and the sale is spoken of again in segment 9.
4. **Feeling and value move onto the gifts.** The words of beauty, desire and joy are strongest around the combs (segment 11) and in the final stage (7.33 per 100 words), at the moment the gifts cannot be used.

**What the reader learns to call "wise".** The narrator names the result only in the last paragraph, where 11 of the 14 wisdom words appear. The reader is led to feel the value of the gifts before being told to call it wisdom, and the earlier comments about calculation prepare the final reversal: the two characters who gave up their most valuable possessions are called "the wisest".

The results also correct the first impression in two ways. The conversion is not one smooth process but several, running at different speeds. And the sale of the body is as prominent in the language as money, and returns in Stage III after the money language has gone.

## 7.10 Position of the reader and role of the model

The story was read by a college student from a busy, multilingual urban Indian setting, with English as an academic language rather than the language of the home. This position shaped what was noticed first: money, family obligation and feeling. Several of the most telling shame words, such as "parsimony", "mendicancy" and "meretricious", belong to a formal literary register that is easy to pass over when reading for plot and feeling. The extraction step made them visible, so the model complemented the reading rather than replacing it. The choice of the four dimensions, however, came from the reading itself. Another reader might begin from different concerns and arrive at a different set of dimensions.

## 7.11 Limitations

- **The method depends on a word list.** Words such as "present", "nice" and "hair" can have several senses, and counting treats all senses alike. One false positive ("just at present", meaning "now", not "gift") was caught with a stoplist, but others may remain.
- **The extraction was done by a language model** from a prompt written for this study. Every phrase was checked against the text, but the choice of which phrases to extract could differ for another model or another reader.
- **The division into 13 segments and 4 stages** was decided by the researcher from changes of scene. A different division would change the stage figures, although the strongest patterns (the fall of shaming and the final peak of inversion) are also visible in the individual segments.
- **The story is short.** A short segment, such as segment 5 with 80 words, can show a high density from a few hits.
- **Counts show vocabulary, not tone.** Irony in particular is only partly captured by phrases.
- **The statistical tests are a guide only**, because hits within a passage are not independent.

## 7.12 Conclusion

**Answer to the research question.** The narrator's language turns economic loss into emotional value gradually, in four steps, and teaches the reader to call the two characters "wise" only at the very end.

1. **Counting and shame come first.** The story opens in money: shame vocabulary is at 5.86 hits per 100 words in Stage I and falls steadily to 0.00 by Stage IV.
2. **The numbers disappear.** Counting words are absent after Stage I.
3. **The hair is sold in the language of goods and then of confession.** The sale (segment 5) and Della's confession (segment 9) are the two peaks of the corporeal vocabulary.
4. **Feeling and value move onto the gifts.** Words of beauty, desire and joy are strongest around the combs (segment 11) and in Stage IV (7.33 per 100 words), at the moment the gifts cannot be used.

**What the reader learns to call "wise".** The narrator names the result in the last paragraph, where 11 of the 14 wisdom words appear: the couple who sacrificed their most valuable possessions for each other are "the wisest" and "the magi". Earlier comments about calculation ("A mathematician or a wit would give you the wrong answer") prepare this reversal.

**Two additional findings.** The original claim is only partly supported: shame and counting are early, as expected, but selling language returns in Stage III and the narrator's comments appear at both the start and the end. The language of sacrifice is also unequal: most of the vocabulary of selling and of bodily reaction belongs to Della.

In [ ]:
def md_to_html(md_text):
    """Markdown converter: headings, paragraphs, bullet and numbered lists, tables, bold and italics."""
    def inline(t):
        t = escape(t)
        t = re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", t)
        return re.sub(r"\*(.+?)\*", r"<i>\1</i>", t)
    out, mode, table = [], None, []

    def close():
        nonlocal mode
        if mode == "ul":
            out.append("</ul>")
        elif mode == "ol":
            out.append("</ol>")
        mode = None

    def flush_table():
        nonlocal table
        if table:
            rws = [[c.strip() for c in r.strip().strip("|").split("|")] for r in table]
            rws = [r for r in rws if not all(set(c) <= set("-: ") for c in r)]
            html = "<table>"
            for i, r in enumerate(rws):
                tag = "th" if i == 0 else "td"
                html += "<tr>" + "".join(f"<{tag}>{inline(c)}</{tag}>" for c in r) + "</tr>"
            out.append(html + "</table>")
            table = []

    for line in md_text.splitlines():
        if line.startswith("|"):
            close()
            table.append(line)
            continue
        flush_table()
        num = re.match(r"^\d+\.\s+(.*)", line)
        if line.startswith("- "):
            if mode != "ul":
                close(); out.append("<ul>"); mode = "ul"
            out.append(f"<li>{inline(line[2:])}</li>")
        elif num:
            if mode != "ol":
                close(); out.append("<ol>"); mode = "ol"
            out.append(f"<li>{inline(num.group(1))}</li>")
        else:
            close()
            if line.startswith("### "):   out.append(f"<h4>{inline(line[4:])}</h4>")
            elif line.startswith("## "):  out.append(f"<h3>{inline(line[3:])}</h3>")
            elif line.startswith("# "):   out.append(f"<h2>{inline(line[2:])}</h2>")
            elif line.strip():            out.append(f"<p>{inline(line)}</p>")
    close()
    flush_table()
    return "\n".join(out)

TITLE = "Reading “The Gift of the Magi” with an LLM: how language turns loss into value"
PAGE = """<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>@@TITLE@@</title>
<style>
body{font-family:Georgia,'Times New Roman',serif;color:#222;background:#fffdf8;max-width:920px;margin:0 auto;padding:2.5rem 1.2rem;line-height:1.55}
h1{font-size:1.8rem;line-height:1.25;color:#1f3a5f;margin-bottom:.4rem}
h2{font-size:1.25rem;margin-top:2.4rem;border-bottom:1px solid #d8d0bf;padding-bottom:.3rem;color:#1f3a5f}
h3{font-size:1.08rem;color:#1f3a5f;margin-bottom:.2rem}
section{margin:2.2rem 0;padding-bottom:1.2rem;border-bottom:1px solid #e3dccb}
.box{background:#f3eee4;border-left:4px solid #8c2f39;padding:.8rem 1.1rem;margin:1rem 0}
.how{color:#555;font-size:.95rem;margin:.2rem 0 .6rem}
.finding{background:#f3eee4;padding:.6rem .9rem;border-radius:4px}
table{border-collapse:collapse;width:100%;font-size:.9rem;margin:.6rem 0}
th{background:#1f3a5f;color:#fff;text-align:left;padding:.45rem}
td{border-bottom:1px solid #e3dccb;padding:.45rem}
small{color:#666}
@media (prefers-color-scheme: dark){body{background:#15171c;color:#e6e3dc}h1,h2,h3{color:#9db8e0}.box,.finding{background:#232730}td{border-color:#333}small,.how{color:#a9a9a9}}
</style></head><body>
<h1>@@TITLE@@</h1>
<small>Hover over any element for the phrases behind it. Use the toolbar at the top right of each chart to zoom or save a PNG.</small>
<div class="box"><b>Research question.</b> @@RQ@@<br><b>Claim tested.</b> @@CLAIM@@</div>
<h2>Visualisations</h2>
@@BODY@@
<h2>Results by stage (hits per 100 words)</h2>
@@STAGETABLE@@
<h2>How the story was divided</h2>
<p><small>Segments and stages were decided by the researcher from shifts of scene, not by equal length.</small></p>
@@SEGTABLE@@
@@ESSAY@@
<h2>Method note</h2>
<p><small>Phrases were extracted by @@MODEL@@ from a prompt written by the researcher, verified word for word against the text, and matched by whole words, longest phrase first, without counting any stretch of text twice. Tests are descriptive: hits within a passage are not independent.</small></p>
</body></html>"""

def build_page(path, with_essay):
    body = ""
    for k, f in enumerate(FIGS, 1):
        chart = f.to_html(full_html=False, include_plotlyjs="cdn" if k == 1 else False, config=CONFIG)
        body += (f"<section><h3>{escape(TITLES[k])}</h3><p class='how'><b>How to read this:</b> {escape(HOW[k])}</p>"
                 f"{chart}<p class='finding'><b>What it shows:</b> {escape(INTERPRETATION[k])}</p></section>\n")
    stage_table = stage_dens.rename(index=lambda d: DIMENSIONS[d]["label"]).round(2).to_html(border=0)
    seg_table = pd.DataFrame([{"Segment": s["id"], "Paragraphs": s["paras"], "Scene": s["note"],
                               "Stage": s["stage"], "Words": s["words"]} for s in segments]).to_html(index=False, border=0)
    page = (PAGE.replace("@@TITLE@@", escape(TITLE)).replace("@@RQ@@", escape(RESEARCH_QUESTION))
                .replace("@@CLAIM@@", escape(CLAIM)).replace("@@BODY@@", body)
                .replace("@@STAGETABLE@@", stage_table).replace("@@SEGTABLE@@", seg_table)
                .replace("@@MODEL@@", escape(MODEL_NAME))
                .replace("@@ESSAY@@", md_to_html(ESSAY) if with_essay else ""))
    open(path, "w", encoding="utf-8").write(page)
    print("Wrote", path)

build_page("docs/visualisations.html", with_essay=False)
build_page("docs/report.html", with_essay=True)

Wrote docs/visualisations.html
Wrote docs/report.html


In [ ]:
!zip -q -r synthetic_reading_gift_of_the_magi.zip corpus partial_reading prompt extraction docs initial_impression.md
try:
    from google.colab import files
    files.download("synthetic_reading_gift_of_the_magi.zip")
except ImportError:
    print("Not running in Colab: the zip file is in the working folder.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>